# Real-IAD 데이터 분석 실습 · OpenCV

**목표:** 5시점 이미지에서 결함이 어느 시점에 표시되는지, 얼마나 작고 어떤 촬영 조건에서 어려운지 설명하고 모델 실험 조건을 정합니다.

첨부 가이드 1장(PDF 2~6페이지)을 구현했습니다. 위에서 아래 순서로 실행하세요. 기본 실행은 **합성 데이터**입니다. 실제 Real-IAD의 품질·통계·모델 성능을 나타내지 않습니다. GPU/LLM/API 키는 필요 없습니다.

학습 순서: 입력·계약 → 메타데이터 → 전수 감사 → 개발 표본 → 품질·마스크 → 전체 분석 → 결과 해석 → 속도 → 팀 교차 검증.

환경 설치와 실제 데이터 준비는 `README.md`를 먼저 읽으세요.

## 0. 환경과 경로

노트북 파일과 `core.py`가 있는 폴더에서 실행합니다. 실제 데이터로 전환할 때는 `USE_DEMO=False`로 바꾸고 아래 ROOT/JSON_DIR 값을 수정합니다. 실행 중 만든 결과가 섞이지 않도록 실험마다 OUTPUT 이름을 바꾸세요.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as DisplayImage

if not Path('core.py').is_file():
    raise RuntimeError('노트북과 core.py가 있는 폴더에서 실행하세요.')
from core import (build_manifest, audit, analyze, image_features, read_rgb,
                  read_mask, components, nearest, project_split, benchmark, figures)
from backend import NAME
from make_demo import generate
get_ipython().run_line_magic('matplotlib', 'inline')

USE_DEMO = True
ROOT = Path('demo_notebook/images') if USE_DEMO else Path('/data/Real-IAD/realiad_1024')
JSON_DIR = Path('demo_notebook/jsons') if USE_DEMO else Path('/data/Real-IAD/realiad_jsons')
OUTPUT = Path('notebook_results_demo') if USE_DEMO else Path('notebook_results_real')
EXPECTED_VIEWS = ('C1','C2','C3','C4','C5')
CATEGORIES = None  # 실제 분석은 ['toothbrush', 'toy']처럼 필요한 제품 유형 지정
SPLITS = ('train','val','validation')
SIZES = (1024,512,256)
SEED = 42
MAX_PRODUCTS = 20  # 0이면 전체 개발 제품. 먼저 20개를 검수합니다.
OUTPUT.mkdir(parents=True, exist_ok=True)
if USE_DEMO and not ROOT.parent.exists():
    generate(ROOT.parent)
print('backend:', NAME, '| python:', sys.version.split()[0])
print('root:', ROOT.resolve(), '| SYNTHETIC DEMO:', USE_DEMO)

## 1. 메타데이터 → 이미지 매니페스트

제품 유형(category), **제품 그룹(sample_key)**, 카메라(view_id)를 구분합니다. 다섯 시점은 같은 sample_key를 공유합니다. 공식 ADEval 패턴으로 파일명을 파싱하고 원본 공식 분할을 보존합니다. 파싱 실패는 임의 추정하지 않습니다.

공식 JSON의 `train/test` 분할을 그대로 쓰는 것이 기본입니다. 다른 릴리스 형식은 README의 CSV 어댑터를 사용하세요.

In [ ]:
manifest = build_manifest(ROOT, JSON_DIR, OUTPUT/'input', CATEGORIES)
display(manifest.head(10))
print('rows:', len(manifest))
display(manifest.groupby(['category','official_split','anomaly_class']).size().rename('images').reset_index())

## 2. 전체 파일 감사

이 단계는 시험 파일도 **읽기 가능 여부와 정합성**을 검사합니다. 상세 결함 분석과 시험 통계 공개는 다음 단계에서 분할을 선택해 제한합니다.

확인: 읽기 오류, 원본/마스크 크기, 파싱·키 중복, 기대 시점 집합, 제품 라벨 충돌, 제품 분할 누수, 파일 및 RGB 픽셀 해시 누수. 미제공 마스크는 정상으로 간주하지 않습니다.

In [ ]:
checked, issues = audit(ROOT, manifest, OUTPUT/'audit', EXPECTED_VIEWS)
display(issues)
display(checked[['category','official_split','status','mask_state']].value_counts().rename('images').reset_index())
print('오류 상세:', (OUTPUT/'audit/integrity_issues.csv').resolve())
# 오류가 있으면 해당 파일과 제품을 수동 검수합니다.
# analyze()는 키·라벨·누수 문제를 발견하면 중단합니다.

## 3. 개발용 분할과 제품 표본

기본은 train/val만 분석합니다. 실제 공식 정상 학습 트랙에서는 불량 개발 이미지가 없어 결함 통계가 비어 있을 수 있습니다. 그 결과를 test로 자동 보충하지 않습니다.

지도학습/GAN 트랙에서 별도 프로젝트 분할이 필요할 때만 아래 옵션을 켜세요. 공식 분할은 별도 열로 남고, 새로운 train/val/test는 제품별로 묶입니다. 미학습 결함 코드는 `HOLDOUT_DEFECTS`로 지정합니다. 제품당 공식 라벨이 충돌하면 분할을 멈춥니다.

In [ ]:
USE_SUPERVISED_PROJECT_SPLIT = False
HOLDOUT_DEFECTS = ()  # 예: ('ZW',)
if USE_SUPERVISED_PROJECT_SPLIT:
    manifest = project_split(manifest, seed=SEED, holdout_defects=HOLDOUT_DEFECTS)
    manifest.to_csv(OUTPUT/'manifest_supervised.csv', index=False, encoding='utf-8-sig')
products = manifest[['category','sample_key','anomaly_class','project_split']].drop_duplicates()
display(products.groupby(['category','project_split','anomaly_class']).size().rename('products').reset_index())
assert (manifest.groupby(['category','sample_key']).project_split.nunique() == 1).all()
print('EDA splits:', SPLITS, '| test 제외:', 'test' not in SPLITS)

## 4. 한 이미지에서 계산 과정을 확인

회색값 = 0.299R + 0.587G + 0.114B (float64, 0~255). 밝기 중앙값, P95−P5 대비, RGB 중 한 채널이라도 250 이상인 포화 비율, 전체 채널 5 이하인 저휘도 비율을 구합니다.

초점 지표는 4이웃 라플라시안 분산입니다. **OpenCV: BORDER_REFLECT_101 / SciPy: mirror**로 경계를 맞췄습니다. 질감과 잡음도 값에 영향을 주므로 이 숫자 하나로 불량을 판단하지 않습니다.

In [ ]:
dev = checked[checked.project_split.isin(SPLITS) & checked.status.eq('ok')]
if dev.empty:
    raise ValueError('읽을 수 있는 개발 이미지가 없습니다. 경로/분할/감사 결과를 확인하세요.')
row = dev.iloc[0]
rgb = read_rgb(ROOT/row.image_path)
print('RGB shape/dtype:', rgb.shape, rgb.dtype)
display(pd.Series(image_features(rgb), name='value').to_frame())
fig, axes = plt.subplots(1,2,figsize=(10,3))
axes[0].imshow(rgb); axes[0].axis('off'); axes[0].set_title(f'{row.category} {row.view_id}')
axes[1].hist((rgb.astype(float) @ np.array([.299,.587,.114])).ravel(),bins=50,range=(0,255))
axes[1].set_xlabel('Gray value [0,255]'); axes[1].set_ylabel('Pixels'); plt.tight_layout(); plt.show()

## 5. 마스크와 8이웃 연결 성분

mask > 0을 양성으로 정의합니다. 대각선에 닿은 픽셀도 같은 성분입니다. 박스는 (x0,y0,x1,y1)이며 x1/y1 끝점은 제외합니다. scikit-image의 row/column 순서를 공통 x/y로 변환합니다.

마스크가 없거나 읽을 수 없으면 정상 마스크로 대체하지 않고 이 실습을 건너뜁니다. 가장 먼저 나타나는 개발 양성 마스크를 확인합니다.

In [ ]:
positive = checked[checked.project_split.isin(SPLITS) & checked.mask_state.eq('positive')]
if positive.empty:
    print('개발 분할에 양성 마스크가 없습니다. 공식 정상 학습 트랙에서는 예상 가능한 결과입니다.')
else:
    pr = positive.iloc[0]
    image = read_rgb(ROOT/pr.image_path)
    mask = read_mask(ROOT/pr.mask_path, image.shape[:2])
    labels, regions = components(mask)
    display(pd.DataFrame([dict(component_id=cid, **reg) for cid,reg in regions]))
    print('defect pixel ratio:', mask.mean())
    fig,axes=plt.subplots(1,3,figsize=(12,3))
    axes[0].imshow(image);axes[0].set_title('RGB')
    axes[1].imshow(mask,cmap='gray');axes[1].set_title('Binary annotation')
    axes[2].imshow(labels,cmap='nipy_spectral');axes[2].set_title('8-connected regions')
    for ax in axes: ax.axis('off')
    plt.tight_layout();plt.show()
    for size in SIZES:
        resized = nearest(labels, size)
        remaining=np.bincount(resized.ravel(), minlength=int(labels.max())+1)
        print(size, 'lost original components:', sum(remaining[cid]==0 for cid,_ in regions), '/', len(regions))

## 6. 공통 제품 표본으로 분석 실행

처음에는 최대 20제품을 category/결함별 round-robin으로 선정합니다. 실제 분석에 사용한 목록은 `selected_manifest.csv`입니다. 양쪽 버전은 동일 매니페스트/seed/크기/기대 시점을 공유해야 합니다.

전체 파일 감사는 매 실행에 포함됩니다. 대용량 실제 데이터에서는 해시·디코딩 때문에 시간이 걸릴 수 있습니다. 분석은 한 번에 한 이미지를 읽으므로 GPU나 전체 데이터의 RAM 적재는 필요 없습니다.

In [ ]:
metrics, product_summary = analyze(ROOT, manifest, OUTPUT/'analysis',
    splits=SPLITS, sizes=SIZES, expected=EXPECTED_VIEWS,
    max_products=MAX_PRODUCTS, seed=SEED)
RESULTS=OUTPUT/'analysis'
display(pd.read_csv(RESULTS/'counts.csv'))
display(product_summary.head())
display(metrics[['category','sample_key','view_id','mask_state','defect_ratio','na_reason']].head(15))

## 7. 결과 읽기: 가시율·작은 결함·촬영 조건

**가시율:** 분자=양성 마스크가 있는 결함 제품, 분모=해당 시점이 있고 마스크 상태를 판정할 수 있는 결함 제품. 미제공/오류는 분모에서 빼고 unknown_masks로 별도 표시합니다.

**해상도:** 원본 성분 라벨 맵을 공통 pixel-center 최근접 규칙으로 1024/512/256 정사각형으로 옮겨 원래 성분이 남았는지 셉니다. 소실률은 모델 성능이 아닙니다.

**위치 지도:** category와 카메라별 2D 좌표를 따로 봅니다. 여러 카메라의 좌표를 같은 물리 위치로 합치지 않습니다.

In [ ]:
display(pd.read_csv(RESULTS/'visibility.csv'))
display(pd.read_csv(RESULTS/'resolution_summary.csv'))
display(pd.read_csv(RESULTS/'normal_quality_by_view.csv').head())
for path in sorted((RESULTS/'figures').glob('*.png')):
    display(DisplayImage(filename=str(path), width=900))

### 20개 제품 수동 검수

기본 결과 그림은 정상/불량에서 순서상 최대 3제품씩만 보여 줍니다. 아래 셀은 선정한 개발 표본의 **모든 제품** 오버레이를 따로 만듭니다. 결함 마스크가 원본에 제대로 정렬되는지 확인하고 검수 결과를 남기세요.

In [ ]:
RUN_ALL_MANUAL_OVERLAYS = False  # 수동 검수할 때 True
if RUN_ALL_MANUAL_OVERLAYS:
    from core import figures
    comp = pd.read_csv(RESULTS/'components.csv')
    vis = pd.read_csv(RESULTS/'visibility.csv')
    resolution = pd.read_csv(RESULTS/'resolution_sensitivity.csv')
    for i,pr in product_summary.iterrows():
        part_m=metrics[(metrics.category==pr.category)&(metrics.sample_key==pr.sample_key)]
        part_c=comp[(comp.category==pr.category)&(comp.sample_key==pr.sample_key)]
        folder=OUTPUT/'manual_review'/f'product_{i:03d}'
        folder.mkdir(parents=True,exist_ok=True)
        figures(ROOT,part_m,part_c,product_summary.iloc[[i]],vis.iloc[:0],resolution.iloc[:0],folder,EXPECTED_VIEWS)
    print('검수 그림:', (OUTPUT/'manual_review').resolve())

## 8. 처리 시간과 구현 정합성

속도는 같은 장비·같은 제품으로 warmup 후 반복합니다. 디코딩 포함/배열 연산만을 구분합니다. 전체 EDA 처리 시간이 아닌 **품질+성분 커널**의 제품당 시간입니다.

양쪽 결과 폴더가 있으면 비교기를 실행합니다. 입력 해시·설정이 다르면 FAIL이며, 성분 ID의 순서는 공통 geometry로 바꿔 비교합니다.

In [ ]:
RUN_BENCHMARK = True
if RUN_BENCHMARK:
    benchmark(ROOT, manifest, OUTPUT/'benchmark', splits=SPLITS, repeats=3, max_products=5)
    display(json.loads((OUTPUT/'benchmark/benchmark_summary.json').read_text()))

# 양쪽 실제 결과 경로를 지정하고 실행하세요.
# from compare_results import compare
# compare('../RealIAD_OpenCV/notebook_results_demo/analysis',
#         '../RealIAD_ScikitImage/notebook_results_demo/analysis',
#         'parity_report.md')

## 9. 분석 근거를 모델 실험으로 연결

수치를 보고 바로 모델을 정하지 말고 다음 비교 실험을 정합니다.

| 분석 질문 | 다음 실험 |
|---|---|
| 256에서 작은 결함 성분이 사라지는가? | 256/512/1024 또는 ROI 입력 비교 |
| 특정 시점만 양성인 제품이 많은가? | 고정 단일 시점 vs 5시점 점수 최대값 |
| 정상 밝기/초점 분포가 시점마다 다른가? | 전역 전처리 vs 시점별 전처리 |
| 결함 위치가 특정 영상 영역에 몰리는가? | category/view별 GAN 마스크 배치 검수 |
| 일부 코드 제품 수가 적은가? | 유형별 표본 수 명시, 희소 유형 분리 평가 |

`all_masks_known=1`인 제품만 '정확히 한 시점 양성'으로 확정합니다. 미제공 마스크가 있으면 n_positive_views는 관측 하한입니다. 시험 데이터는 설계 선택·GAN 원본·임계값 튜닝에 쓰지 않습니다.

In [ ]:
print((RESULTS/'decisions.md').read_text(encoding='utf-8'))
print('결과 경로:', RESULTS.resolve())
print('다음: 위 리뷰 양식에 근거/가설/담당자를 작성하고 양쪽 parity_report.md를 함께 인수하세요.')